In [1]:
import pygame
def play_music(name):
    pygame.mixer.init()
    pygame.mixer.music.load(name)
    pygame.mixer.music.play()


pygame 2.6.1 (SDL 2.28.4, Python 3.12.5)
Hello from the pygame community. https://www.pygame.org/contribute.html


Dataset cleaning

In [2]:
from datasets import load_dataset
import re

dataset = load_dataset("wikitext", "wikitext-103-raw-v1")

def clean(example):
    text = example["text"]

    # remove wiki headings
    text = re.sub(r"=+ .*? =+", "", text)
    text = re.sub(r"@.@" , "", text)

    # whitespace cleanup
    text = re.sub(r"\s+", " ", text).strip()

    return {"text": text}
dataset = dataset.map(clean)

cleaned_dataset = dataset["train"].filter(
    lambda x: len(x["text"]) > 50
)

c:\Users\Gaurav B V\anaconda3\envs\bot\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from transformers import BertTokenizer
from torch.utils.data import DataLoader

# Load the standard BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function(examples):
    # Padding and truncation are essential for batching
    return tokenizer(
        examples["text"], 
        padding="max_length", 
        truncation=True, 
        max_length=128 # You can increase this to 256 to match your config
    )

# Apply tokenization to all splits (train, test)
# tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets = cleaned_dataset.map(tokenize_function,batched=True)

Model importing


In [4]:
import torch
import torch.nn as nn
from transformers import BertConfig
from tqdm.auto import tqdm
from DynamicActivationBertModel import MyBertModel

config = BertConfig(
    vocab_size=30522,
    hidden_size=256,
    num_hidden_layers=6,
    num_attention_heads=4,
    intermediate_size=128,
    max_position_embeddings=256,
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1,
)

model = MyBertModel(config)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Parameters: 17,704,524


In [5]:
import torch
import torch.nn as nn
from torch.optim import AdamW

# 1. Device configuration (CUDA, MPS for Mac, or CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
model.to(device)

# 2. Loss Function (CrossEntropy for Masked Language Modeling or Classification)
criterion = nn.CrossEntropyLoss().to(device)

# 3. Optimizer (Weight decay is crucial for Transformers)
optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)

: 

Experimentation Training loop


In [ ]:
from torch.utils.data import DataLoader, TensorDataset
from transformers import get_linear_schedule_with_warmup


vocab_size = 30522
mask_token_id = 103  
sample_size = 100000
epochs = 20
MASK_PROB = 0.15
BATCH_SIZE = 4

test_test = tokenized_datasets[0:]["input_ids"]

optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
total_steps = sample_size * epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps*0.1,
    num_training_steps=total_steps*epochs
)

# --- Prepare DataLoader once, outside the loop ---
input_ids = torch.tensor(tokenized_datasets[0:sample_size]["input_ids"])  # (N, seq_len)
dataset = TensorDataset(input_ids)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

model.train()

for epoch in range(epochs):
    total_loss = 0

    for (batch,) in dataloader:
        optimizer.zero_grad()

        data = batch.to(device)             # (B, seq_len)
        labels = data.clone()               # (B, seq_len)

        # --- Masking ---
        probability_matrix = torch.full(labels.shape, MASK_PROB)
        masked_indices = torch.bernoulli(probability_matrix).bool()  # (B, seq_len)
        inputs = data.clone()
        inputs[masked_indices] = mask_token_id                        # (B, seq_len)

        # --- Forward pass ---
        outputs = model(inputs)             # (B, seq_len, vocab_size)

        # --- Fix: index correctly into 3D output ---
        loss = criterion(
            outputs[masked_indices],        # (num_masked, vocab_size)
            labels[masked_indices]          # (num_masked,)
        )

        total_loss += loss.item()
        loss.backward()
        optimizer.step()
        scheduler.step()

    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch} | Total Loss: {total_loss:.4f} | Avg Loss: {avg_loss:.4f}")

# play_music("01 King Gnu - SPECIALZ.flac")

Validation of the learning

In [ ]:
from datasets import load_dataset
from transformers import BertTokenizer
from torch.utils.data import DataLoader

ds = load_dataset("sentence-transformers/stsb")


tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function_text(text):
    return tokenizer(
        text,
        padding="max_length", 
        truncation=True, 
        max_length=128 
    )

In [ ]:
validation_inputs = []
for i in ds["train"]:
    i["sentence1_tokens"] = tokenize_function_text(i["sentence1"])
    i["sentence2_tokens"] = tokenize_function_text(i["sentence2"])
    validation_inputs.append(i)


In [2]:
import torch
import torch.nn.functional as F
import gc
def sentance_embeddings(text):    
    raw_input_ids = text
    input_tensor1 = torch.tensor(raw_input_ids).unsqueeze(0).to(device)
    output1 = model(input_tensor1,None,True)
    del input_tensor1
    gc.collect()
    return  output1

def cosine_similarity(embeddings,embeddings2):
# embedding shape: [1, 128, 256]
    # embeddings = embeddings.mean(dim=1)   # [1,256]
    # embeddings2 = embeddings2.mean(dim=1)
    print(embeddings.shape)
    sim = F.cosine_similarity(embeddings, embeddings2)
    return sim


In [ ]:
import torch
import torch.nn as nn
from torch.optim import AdamW
import math

# 1. Device configuration (CUDA, MPS for Mac, or CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
loss = 0
for i in validation_inputs:
    raw_input_ids = i['sentence1_tokens']['input_ids']
    output1 = sentance_embeddings(raw_input_ids)
    output2 = sentance_embeddings( i['sentence2_tokens']['input_ids'])

    # input_tensor1 = torch.tensor(raw_input_ids).unsqueeze(0).to(device)
    print(output2.shape)
    # output1 = model(input_tensor1,None,True)
    sim = cosine_similarity(output1,output2)
    print(sim,i['score'])
    
    loss+= abs(sim-i['score'])
    print(output2.shape)
    print(sim.item())
